# 01 — TRIBE verification (decision gate 17)

Track A, Colab GPU. Verifies the pinned checkpoint against
`config/checkpoints.lock`, reproduces Meta's published example, records the
environment, and writes `outputs/verification/gate17.json`.

**`scripts/run_tribe_inference.py` refuses to touch a project stimulus until
that artifact exists for the pinned revision.** Gate 17 is enforced in code,
not on a checklist. It is a *project* gate, separate from HuggingFace access:
having Llama-3.2-3B approved is what makes this notebook runnable, not what
makes it unnecessary.

Nothing below computes anything itself — every cell is a bootstrap step or a
call into `scripts/`.

## Bootstrap

1. **GPU check** — Track A needs one. Runtime → Change runtime type → GPU.
2. **Clone `phase2-tribe` and install**, then **Runtime → Restart session**.
   The branch is not optional: the repo's default branch has no `src/tribe/`.
3. **Session setup** — `setup_session()` mounts Drive, derives every path, sets
   `HF_HOME`, and authenticates from Colab Secrets (🔑).

Step 3 re-derives everything it needs and is safe to run twice, so it is what
you re-run after a restart or a dropped connection — nothing above it is needed
again.

It lives in `src/colab/session.py`, not in this cell. Three notebooks need the
same paths, and a copy-pasted cell that decides where 13 GB goes is exactly the
kind of thing that drifts. Step 2 is the one part that cannot move into the
repo: it is what clones the repo.

### Where things are written

| | goes to | why |
|---|---|---|
| predictions, gate handoff | **Drive** | GPU time against a gated model; not re-fetchable |
| model weights (`HF_HOME`) | **ephemeral disk** | ~13 GB, re-downloads in about a minute |

`HF_HOME` must be set *before* any HuggingFace import in the process — the
cache location is fixed at import time and cannot be moved afterwards.
`setup_session` enforces that in code rather than by cell ordering: it raises if
the hub is already pointed somewhere else, and is a no-op if it agrees.

The weights are ~13 GB against a 15 GB Drive quota and came down at ~250 MB/s;
keeping them would consume the space the predictions need in order to save
about a minute per session. Drive also cannot hold symlinks, so
`huggingface_hub` stores every weight file twice there.

**One-time cleanup** if you ran an earlier session, which did put them on Drive:

```
!du -sh /content/drive/MyDrive/NeuroTutorSim/*
!rm -rf /content/drive/MyDrive/NeuroTutorSim/hf
```

Everything after that is a single call into a script in `scripts/`. No project
logic lives in this notebook: a cell dies with the session, and brief §4.3
requires every result to come from a script in the repo.

In [ ]:
# 1. GPU check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

In [ ]:
# 2. Clone the repo and install the pinned Track A stack into Colab's interpreter
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
# Not optional. Phase II lives on `phase2-tribe`; the default branch (`master`)
# has no src/tribe/ and no scripts/run_tribe_*.py, so an unpinned clone fails
# further down with a confusing "No such file or directory".
BRANCH = "phase2-tribe"
REPO_DIR = Path("/content/neurotutorsim")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# --system installs into Colab's own interpreter. `uv sync` would build a venv
# this kernel cannot import from.
#
# `.[tribe,dev]`, NOT `--extra tribe --extra dev`: in pip mode uv rejects --extra
# unless the target is `-r <file>`. With `-e .` it exits 2 with
#   "Requesting extras requires a ... pyproject.toml ... Use <dir>[extra] instead"
subprocess.run(
    ["uv", "pip", "install", "--system", "-e", ".[tribe,dev]"],
    cwd=REPO_DIR,
    check=True,
)

print("installed; Runtime -> Restart session, then run the cell below")

### ⚠️ Runtime → Restart session now

Then continue from the cell below. It re-derives every path it needs, so
nothing above has to run again — and it is also the first cell to re-run after
a disconnect.

In [ ]:
# 3. Session setup: mount Drive, derive every path, set HF_HOME, authenticate.
#    This is the cell to run first after a restart or a dropped connection, and
#    it is safe to run twice.
#
#    The logic is in src/colab/session.py, not here: three notebooks need the
#    same paths, and a cell that decides where 13 GB goes must not be something
#    that can drift between copies.
import sys

sys.path.insert(0, "/content/neurotutorsim")
from src.colab import setup_session

paths = setup_session()

# Bound for the cells below; every one of them comes from the same call.
REPO_DIR, CACHE_ROOT, HANDOFF = paths.repo_dir, paths.cache_root, paths.handoff

## Re-pinning the revision — normally skipped

**The revision is already pinned and committed.** `config/tribe.yaml` names the
SHA and `config/checkpoints.lock` holds its checksums, so a fresh clone arrives
ready to verify. **Skip to *Run gate 17*.**

The three cells below exist for one situation: deliberately moving the pin to a
different checkpoint revision. Read the SHA off the Hub, write it into this
session's config, record the new checksums. `--write-lock` refuses to overwrite
an existing entry, so a re-pin is also a decision about the old one.

**The clone is ephemeral, so an edit here is not the pin of record.** The last
section copies the files to Drive; a new pin counts only once committed from a
machine with push access.

In [ ]:
# Print the checkpoint's current commit SHA. Nothing is written.
!cd "${REPO_DIR:?run the session setup cell first}" && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --resolve-revision

In [ ]:
# Paste the SHA printed above, then run. This rewrites `checkpoint_revision` in
# this session's copy of config/tribe.yaml: one key, matched on its own line, so
# a malformed edit fails here instead of surfacing later as a wrong cache key.
import re
import sys
from pathlib import Path

REVISION_SHA = ""   # <- 40 lowercase hex characters, from the cell above

assert re.fullmatch(r"[0-9a-f]{40}", REVISION_SHA), (
    "REVISION_SHA must be the 40-character lowercase hex SHA printed above"
)

config_path = Path("/content/neurotutorsim/config/tribe.yaml")
patched, n = re.subn(
    r"(?m)^checkpoint_revision:.*$",
    f'checkpoint_revision: "{REVISION_SHA}"',
    config_path.read_text(encoding="utf-8"),
)
if n != 1:
    raise SystemExit(f"expected exactly one checkpoint_revision line, matched {n}")
config_path.write_text(patched, encoding="utf-8")

# Read it back through the real loader -- that is the check that the pin is valid.
sys.path.insert(0, "/content/neurotutorsim")
from src.tribe.config import load_config

print("pinned:", load_config(config_path).require_resolved_revision())

In [ ]:
# Record the checkpoint's file checksums for the revision just pinned.
# A separate, explicit step: a lock that writes itself verifies nothing. It also
# refuses to overwrite an existing entry -- a changed checksum for a pinned
# revision means either the pin or the download is wrong, and both need a human.
!cd "${REPO_DIR:?run the session setup cell first}" && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --write-lock

## Run gate 17

Checksums the pinned checkpoint against the lock, reproduces Meta's published
example, asserts the output mesh is fsaverage5 (20484 vertices), records the
environment, and writes the gate artifact.

In [ ]:
!cd "${REPO_DIR:?run the session setup cell first}" && python scripts/run_tribe_verification.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT"

## Before closing the session

Four files are the evidence that gate 17 was cleared on this hardware, and all
four live in the ephemeral clone. The cell below copies them to
`MyDrive/NeuroTutorSim/gate17/`.

**What each one is for — they do not all have the same destination.**

| file | in git? | what to do with it |
|---|---|---|
| `outputs/verification/gate17.json` | **no** — `outputs/` is gitignored | **Nothing.** Leave it on Drive. It attests that *this machine* cleared the gate, so it is deliberately not source. Notebook 02's preflight copies it back into the clone automatically; that is its only consumer. |
| `config/tribe.yaml` | **yes** — already committed | **Nothing, normally.** The repo already carries the pin, and notebook 02 prefers a committed pin over the Drive copy. Only if you *re-pinned* above: download it, replace the repo file, commit. |
| `config/checkpoints.lock` | **yes** — already committed | **Nothing, normally.** Same as above, and it travels with `tribe.yaml` — never commit one without the other or `verify_checkpoint` fails on every machine. |
| `docs/tribe_environment.md` | **yes** — not committed yet | **Download it, put it in `docs/`, commit it.** This is the §6.1 item 15 environment record: GPU, CUDA, Python and package versions at the moment the gate passed. Regenerate and re-commit only when the hardware or the pinned stack changes — a timestamp-only diff is noise. |

The rule behind the table: **anything that describes the *project* goes to git;
anything that describes *this run on this VM* stays on Drive.** The pin and the
environment record are the project. The gate artifact is the run.

Committing has to happen from a machine with push access — the Colab clone has
no credentials, by design.

In [ ]:
import shutil
from pathlib import Path

# Everything that has to outlive the session. The gate artifact is first
# because notebook 02 refuses to start without it.
HANDOFF_FILES = (
    "outputs/verification/gate17.json",
    "config/tribe.yaml",
    "config/checkpoints.lock",
    "docs/tribe_environment.md",
)

for relative in HANDOFF_FILES:
    source = REPO_DIR / relative
    if source.exists():
        shutil.copy2(source, HANDOFF / source.name)   # flat: notebook 02 reads by name
        print(f"copied  {relative}  ->  {HANDOFF / source.name}")
    else:
        print(f"MISSING {relative}  (gate 17 did not get that far)")

environment = REPO_DIR / "docs" / "tribe_environment.md"
if environment.exists():
    print()
    print(environment.read_text(encoding="utf-8")[:800])